In [2]:
!pip install git+https://github.com/cayleypy/cayleypy.git@e1518b6 --no-deps

In [3]:
import sys
import time
from math import ceil
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from cayleypy import CayleyGraph, PermutationGroups
from torch import nn

lib_name = "pilgrim"

def add_repo_src_to_path(marker_files=("pyproject.toml", ".git"), marker_dirs=("src",)):
    """
    Add <repo>/src to sys.path by walking upward from the notebook location (cwd).

    Works when the notebook is in a subfolder like notebooks/.
    """
    here = Path.cwd().resolve()
    if '/kaggle' in str(here) and Path(f'/kaggle/input/{lib_name}').is_dir():
        sys.path.insert(0, str(Path(f'/kaggle/input/{lib_name}')))
        return "kaggle", str(Path(f'/kaggle/input/{lib_name}'))
        

    for p in [here, *here.parents]:
        # Heuristic: if folder has src/ OR looks like repo root
        has_src = (p / "src").is_dir()
        looks_like_repo = any((p / m).exists() for m in marker_files) or any((p / d).is_dir() for d in marker_dirs)

        if has_src and looks_like_repo:
            sys.path.insert(0, str(p / "src"))
            return "local", str(p / "src")

        # Even if no markers, accept first parent that has src/
        if has_src:
            sys.path.insert(0, str(p / "src"))
            return "local", str(p / "src")

    return "none", ""

where, src_path = add_repo_src_to_path()
print("import root:", where, src_path)

from pilgrim.model import AlPilgrim, GetModel, Pilgrim, SimpleMLP  # noqa: E402
from pilgrim.utils.graph_utils import half_interleave, identity  # noqa: E402
from pilgrim.utils.training_utils import (  # noqa: E402
    plot_train_val_loss,
    train_model_one_n,
    try_beam,
)

In [4]:
def perm_like_5(n: int) -> np.ndarray:
    """
    Generalizes permutation #5 pattern to odd n.
    
    For even i: pair evens by shifting by m=(n+1)//2:
        i < m  -> i+m
        i >= m -> i-m
        
    For odd i:
      - if n % 4 == 3: keep 1 fixed, swap (3<->5), (7<->9), ...
    
    Returns:
        np.ndarray of shape (n,), permutation p with p[i] = image of i.
        
    """
    if n <= 0 or n % 2 == 0:
        raise ValueError("n must be a positive odd integer")
    if n % 4 != 3:
        raise ValueError("n must be 3 modulo 4")

    p = np.empty(n, dtype=np.int64)
    m = (n + 1) // 2

    for i in range(n):
        if i % 2 == 0:
            # even indices: shift by m within evens
            p[i] = i + m if i < m else i - m
        elif n % 4 == 3:
            # odd indices: 1 fixed, others swap by ±2 depending on mod 4
            if i == 1:
                p[i] = 1
            elif i % 4 == 3:
                p[i] = i + 2
        else:
            # i % 4 == 3
            p[i] = i - 2

    return p

In [5]:
CFG = {
    'list_beam_width': [2**12]*2 + [2**13]*2 + [2**15]*2 + [2**16]*2,
    'central_state_type': 'standard',
    'initial_state_type': 'reverse',

    'hd1': 1024,
    'residual_blocks': [2048]*2 + [1024]*2 + [512]*2,
    'dropout_rate': 0.25,
    'weight_decay': 0.025,
    "lr": 1e-3,
    "num_epochs": 400,
    "lr_scheduler": {"type": "cosine_restarts", "t0": 20, "t_mult": 3, "eta_min": 1e-5},
    "val_ratio": 0.1,
    "batch_size": 16384,

    # loss regularization
    "lip_weight": 0.025,
    "lip_max_states": 1024, # from batch
    "lip_generator_indices": None,
    "lip_max_generators": 2,
    "lip_seed": None,
    "lip_state_batch_size": 4096,
    "lip_reduction": "mean",
    "lip_val_metric": True,

    "rw_width": 2500,
    "rw_mode": "nbt",

    'history_depth': 0,

    # Will be overridden in training
    "num_classes": 17,
    "state_size": 17,
    "rw_length": 17*(17+5)//(4*(4-1)) + 30,
    "beam_max_steps": 17*17
}

In [7]:
rows = []
histories = {}

n = 55

for lr0, t0, mult, num_epochs in [
    (1e-3, 0, 0, 300),
]:

    central = identity(n)
    try:
        start = perm_like_5(n)
    except Exception:
        continue
    print(f"\n=== n={n}===")

    graph = CayleyGraph(
        PermutationGroups.pancake(n).make_inverse_closed().with_central_state(central),
        dtype=torch.int8,
        batch_size=2**17
    )

    CFG_local = dict(CFG)
    CFG_local["num_classes"] = n
    CFG_local["state_size"] = n
    CFG_local["rw_length"] = ceil(17/14*n)
    CFG_local["beam_max_steps"] = n*n
    CFG_local["input_encoder_type"] = "embedding_flatten"
    CFG_local["embedding_dim"] = n*2
    CFG_local["model_dtype"] = torch.float32

    CFG_local['lr'] = lr0
    CFG_local['num_epoch'] = num_epochs
    if t0 == 0:
        CFG_local["lr_scheduler"] = None
    else:
        CFG_local["lr_scheduler"] = {"type": "cosine_restarts", "t0": t0, "t_mult": mult, "eta_min": 1e-6}

    model = AlPilgrim(CFG_local)

    tic = time.perf_counter()

    train_history = train_model_one_n(
        CFG_local, 
        model,
        graph,
        print_every=25,
        return_history=True, 
        track_grad_norm=True,
    )
    toe = time.perf_counter()

    print(f"  training time: {(tic-toe)/1000000} s")

    _ = plot_train_val_loss(train_history, 
        title=(
            f"n={n}\n"+
            f"residuals: {CFG_local['residual_blocks']}   lip_weight={CFG_local['lip_weight']}",
        ),
        y_scale="adaptive",
    )
    plt.show()
    histories[f'{lr0}_{t0}_{mult}_{num_epochs}'] = train_history

    import pickle
    with open('histories.pkl', 'wb') as f:
        pickle.dump(histories, f)

    best_d, best_bw, history_bw = try_beam(CFG_local, graph, model, start)

In [ ]:
with torch.no_grad():
    out = model.input_encoder(torch.tensor(np.array([central]), device=graph.device)).cpu().detach().numpy()

## Check if swap of 2 positions is the same as the swap of the output embeddings

In [ ]:
import torch
import seaborn as sns
import matplotlib.pyplot as plt

@torch.no_grad()
def encode_matrix(encoder, z):
    e = encoder(z)[0]                       # (state_size*emb_dim,)
    state_size = z.size(1)
    emb_dim = e.numel() // state_size
    return e.view(state_size, emb_dim).detach().cpu()  # (state_size, emb_dim)

def swap_positions(z, i, j):
    z2 = z.clone()
    z2[:, i], z2[:, j] = z2[:, j].clone(), z2[:, i].clone()
    return z2

def plot_swap_heatmaps(encoder, z, i, j):
    E  = encode_matrix(encoder, z)
    E2 = encode_matrix(encoder, swap_positions(z, i, j))

    v = float(torch.max(torch.abs(torch.stack([E, E2]))))
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    sns.heatmap(E,  ax=axes[0], cmap="coolwarm", center=0, vmin=-v, vmax=v)
    axes[0].set_title("E (original): rows=positions, cols=emb dims")
    axes[0].set_xlabel("emb dim"); axes[0].set_ylabel("position")

    sns.heatmap(E2, ax=axes[1], cmap="coolwarm", center=0, vmin=-v, vmax=v)
    axes[1].set_title(f"E (swap positions {i}<->{j})")
    axes[1].set_xlabel("emb dim"); axes[1].set_ylabel("position")

    sns.heatmap(E2 - E, ax=axes[2], cmap="coolwarm", center=0)
    axes[2].set_title("E_swap - E (difference)")
    axes[2].set_xlabel("emb dim"); axes[2].set_ylabel("position")

    plt.tight_layout()
    plt.show()

# usage:
z = torch.tensor(np.array([central]), device=graph.device)
plot_swap_heatmaps(model.input_encoder, z, 3, 7)